# Phase 12 — MoE-Labor: Ist Router-Kohärenz ein Selektions-Artefakt?

Testet den Artefakt-Verdacht gegen Cell 27A an einem selbst trainierten
Spielzeug-MoE (32 Experten, top-4, 4 Layer, TinyShakespeare, kein Köder).
Drei Läufe (Muon×2 Seeds, Adam), drei Gruppenarten: strukturiert ko-aktiv,
zufalls-positions-ko-aktiv (der eigentliche Artefakt-Test) und zufällige
Experten. Nutzt `SingleDeviceTurboMuonWithAuxAdam` aus dem geklonten
flash-newton-schulz-Repo. **GPU empfohlen** — Laufzeit ~10–20 min.

In [ ]:
# === MoE-Labor — ist Router-Kohaerenz ein Selektions-Artefakt? ==============
# Offene Wunde aus Cell 27A: die Router-Zeilen der 317 sind untereinander
# kohaerenter als eine Zufallsgruppe (+7.8%, 32/40 Layer, p=0.0002). ABER die
# 317 wurden ueber KO-AKTIVIERUNG gefunden - gemeinsam feuernde Experten
# koennten konstruktionsbedingt aehnliche Antennen haben. Das ist am 35B nicht
# testbar (kein Retraining), hier schon: winziges MoE-GPT, harmloser Text,
# kein Koeder, drei Trainingslaeufe - dann exakt die Cell-27A-Metrik auf
# DREI Gruppenarten:
#   STRUKTUR    top-G Experten, die auf einer STRUKTURIERTEN Positionsklasse
#               ko-aktivieren (hier: Zeilenenden) - Analogon zu den 317
#   ZUFALLS-POS top-G Experten, die auf einer ZUFAELLIGEN Positionsmenge
#               gleicher Groesse ko-aktivieren  <- der Artefakt-Test
#   ZUFALLS-EXP G zufaellige Experten                <- Kontrolle wie Cell 27A
# Lesart: Ist ZUFALLS-POS genauso kohaerent wie STRUKTUR, ist Kohaerenz eine
# Folge des Auswahlverfahrens (Artefakt). Bleibt ZUFALLS-POS auf Kontroll-
# niveau, ueberlebt der Cell-27A-Befund.
# Sekundaer: Muon (Turbo) vs. Adam - haengt die Kohaerenz am Optimierer?
# Braucht GPU (laeuft auch auf CPU, dann sehr langsam).
import os, math, subprocess, urllib.request, itertools, sys
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
REPO="/content/flash-newton-schulz"
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","--depth","1",
                    "https://github.com/Erikiss/flash-newton-schulz",REPO],check=True)
sys.path.insert(0,REPO)
from turbo_muon_torch import SingleDeviceTurboMuonWithAuxAdam
D_MODEL=128; N_HEAD=4; N_LAYER=4; N_EXP=32; TOP_K=4; D_FF=128
BLOCK=128; BATCH=32; STEPS=1500; GROUP=6; SEEDS=(0,1)
dev="cuda" if torch.cuda.is_available() else "cpu"
# ---------------- pure Logik (testbar) --------------------------------------
def top_experts(counts,G,exclude=()):
    """G haeufigste Experten (Index-Array), stabil, ohne exclude"""
    c=np.asarray(counts,float).copy()
    for e in exclude: c[e]=-1
    return sorted(np.argsort(-c,kind="stable")[:G].tolist())
def coherence(rows,ids):
    """Median |cos| ueber alle Paare der Router-Zeilen in ids"""
    R=np.asarray(rows,float)[list(ids)]
    R=R/np.maximum(np.linalg.norm(R,axis=1,keepdims=True),1e-12)
    M=np.abs(R@R.T); iu=np.triu_indices(len(ids),1)
    return float(np.median(M[iu])) if len(ids)>1 else float("nan")
def sign_p(nplus,n):
    if n==0: return 1.0
    t=min(nplus,n-nplus)
    tail=sum(math.comb(n,i) for i in range(0,t+1))/2**n
    return min(1.0,2*tail)
def verdict_lab(p_struct,r_struct,p_rand,r_rand,alpha=0.05):
    """p/r: Vorzeichentest-p und medianes Verhaeltnis gegen ZUFALLS-EXP"""
    s=(p_struct<alpha and r_struct>1.0); z=(p_rand<alpha and r_rand>1.0)
    if s and z: return "ARTEFAKT"
    if s and not z: return "ECHT"
    if z and not s: return "INVERS"
    return "KEIN-EFFEKT"
# ---------------- Daten (Zeichen-Ebene, kein Tokenizer noetig) --------------
URL="https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
txt=urllib.request.urlopen(URL).read().decode("utf-8")
chars=sorted(set(txt)); stoi={c:i for i,c in enumerate(chars)}; V=len(chars)
data=torch.tensor([stoi[c] for c in txt],dtype=torch.long)
n=int(0.9*len(data)); train,val=data[:n],data[n:]
NL=stoi["\n"]
print("Daten: %d Zeichen, %d Symbole | Geraet: %s"%(len(data),V,dev))
def batch(split,bs=BATCH,g=None):
    d=train if split=="train" else val
    ix=torch.randint(len(d)-BLOCK-1,(bs,),generator=g)
    x=torch.stack([d[i:i+BLOCK] for i in ix])
    y=torch.stack([d[i+1:i+1+BLOCK] for i in ix])
    return x.to(dev),y.to(dev)
# ---------------- Modell ----------------------------------------------------
class MoE(nn.Module):
    def __init__(s):
        super().__init__()
        s.gate=nn.Linear(D_MODEL,N_EXP,bias=False)
        s.w1=nn.Parameter(torch.randn(N_EXP,D_MODEL,D_FF)*0.02)
        s.w2=nn.Parameter(torch.randn(N_EXP,D_FF,D_MODEL)*0.02)
    def forward(s,x,collect=None):
        B,T,Dm=x.shape; xf=x.reshape(-1,Dm)
        p=torch.softmax(s.gate(xf),-1)
        w,idx=p.topk(TOP_K,-1); w=w/w.sum(-1,keepdim=True)
        out=torch.zeros_like(xf)
        for e in range(N_EXP):
            hit=(idx==e)
            if not hit.any(): continue
            r,sl=hit.nonzero(as_tuple=True)
            h=torch.relu(xf[r]@s.w1[e])@s.w2[e]
            out.index_add_(0,r,h*w[r,sl].unsqueeze(-1))
        if collect is not None: collect.append(idx.detach().reshape(B,T,TOP_K).cpu())
        return out.view(B,T,Dm)
class Block(nn.Module):
    def __init__(s):
        super().__init__()
        s.ln1=nn.LayerNorm(D_MODEL); s.ln2=nn.LayerNorm(D_MODEL)
        s.attn=nn.MultiheadAttention(D_MODEL,N_HEAD,batch_first=True)
        s.moe=MoE()
    def forward(s,x,collect=None):
        h=s.ln1(x); T=x.shape[1]
        m=torch.triu(torch.ones(T,T,device=x.device,dtype=torch.bool),1)
        a,_=s.attn(h,h,h,attn_mask=m,need_weights=False)
        x=x+a
        return x+s.moe(s.ln2(x),collect)
class TinyMoE(nn.Module):
    def __init__(s):
        super().__init__()
        s.emb=nn.Embedding(V,D_MODEL); s.pos=nn.Embedding(BLOCK,D_MODEL)
        s.blocks=nn.ModuleList([Block() for _ in range(N_LAYER)])
        s.ln=nn.LayerNorm(D_MODEL); s.head=nn.Linear(D_MODEL,V,bias=False)
    def forward(s,x,y=None,collect=None):
        T=x.shape[1]
        h=s.emb(x)+s.pos(torch.arange(T,device=x.device))[None]
        for i,b in enumerate(s.blocks):
            h=b(h,None if collect is None else collect.setdefault(i,[]))
        lg=s.head(s.ln(h))
        loss=None if y is None else F.cross_entropy(lg.reshape(-1,V),y.reshape(-1))
        return lg,loss
def make_opt(model,kind):
    mu=[p for nm,p in model.named_parameters() if p.ndim>=2 and "emb" not in nm and "head" not in nm]
    aux=[p for nm,p in model.named_parameters() if not (p.ndim>=2 and "emb" not in nm and "head" not in nm)]
    if kind=="adam":
        return torch.optim.AdamW(model.parameters(),lr=3e-4,betas=(0.9,0.95))
    return SingleDeviceTurboMuonWithAuxAdam([
        dict(params=mu,use_muon=True,lr=0.02,momentum=0.95,weight_decay=0,variant="turbo"),
        dict(params=aux,use_muon=False,lr=3e-4,betas=(0.9,0.95),eps=1e-10,weight_decay=0)])
def train_one(kind,seed):
    torch.manual_seed(seed); np.random.seed(seed)
    g=torch.Generator().manual_seed(seed)
    m=TinyMoE().to(dev); opt=make_opt(m,kind)
    for st in range(STEPS):
        x,y=batch("train",g=g)
        _,loss=m(x,y); opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        if (st+1)%500==0:
            with torch.no_grad():
                xv,yv=batch("val",g=g); _,vl=m(xv,yv)
            print("    %s/seed%d  Schritt %4d  train %.3f  val %.3f"%(kind,seed,st+1,loss.item(),vl.item()))
    return m
@torch.no_grad()
def collect_counts(m,n_batch=8,seed=123):
    """Aktivierungszaehlung je Layer: (a) an Zeilenenden, (b) an Zufallspositionen"""
    g=torch.Generator().manual_seed(seed)
    cs=np.zeros((N_LAYER,N_EXP)); cr=np.zeros((N_LAYER,N_EXP)); n_struct=0
    rng=np.random.default_rng(seed)
    for _ in range(n_batch):
        x,y=batch("val",g=g); col={}
        m(x,collect=col)
        struct=(y.cpu().numpy()==NL)                       # naechstes Zeichen = Zeilenende
        n_struct+=int(struct.sum())
        flat=struct.reshape(-1)
        randmask=np.zeros_like(flat);
        pick=rng.choice(len(flat),size=int(flat.sum()),replace=False)
        randmask[pick]=True
        for l in range(N_LAYER):
            idx=col[l][0].numpy().reshape(-1,TOP_K)
            for e in range(N_EXP):
                hit=(idx==e).any(1)
                cs[l,e]+=int((hit&flat).sum()); cr[l,e]+=int((hit&randmask.astype(bool)).sum())
    return cs,cr,n_struct
# ---------------- Laeufe ----------------------------------------------------
# ---------------- Rauchtest: greift die Mechanik? ---------------------------
def smoke():
    torch.manual_seed(0); g=torch.Generator().manual_seed(0)
    m=TinyMoE().to(dev)
    x,y=batch("train",bs=4,g=g)
    lg,loss=m(x,y)
    assert lg.shape==(4,BLOCK,V) and torch.isfinite(loss), "Forward defekt"
    loss.backward()
    n_grad=sum(1 for p in m.parameters() if p.grad is not None and torch.isfinite(p.grad).all())
    assert m.blocks[0].moe.w1.grad is not None and m.blocks[0].moe.gate.weight.grad is not None, \
        "keine Gradienten an den Experten/Routern - MoE-Dispatch defekt"
    col={}
    with torch.no_grad(): m(x,collect=col)
    idx=col[0][0]
    assert idx.shape==(4,BLOCK,TOP_K) and int(idx.max())<N_EXP, "collect defekt"
    used=len(torch.unique(idx))
    for kind in ("muon","adam"):
        mm=TinyMoE().to(dev); op=make_opt(mm,kind)
        _,l2=mm(x,y); l2.backward(); op.step()
        assert all(torch.isfinite(p).all() for p in mm.parameters()), "%s-Schritt erzeugt NaN"%kind
    print("RAUCHTEST ok | Verlust %.3f | %d/%d Parametertensoren mit endlichem Gradienten | %d/%d Experten aktiv"
          %(loss.item(),n_grad,sum(1 for _ in m.parameters()),used,N_EXP))
smoke()
RUNS=[("muon",SEEDS[0]),("muon",SEEDS[1]),("adam",SEEDS[0])]
COH={"STRUKTUR":[],"ZUFALLS-POS":[],"ZUFALLS-EXP":[]}
BYOPT={}
print("\nTRAINING (%d Schritte je Lauf, %d Experten top-%d, Gruppe G=%d):"%(STEPS,N_EXP,TOP_K,GROUP))
for kind,seed in RUNS:
    m=train_one(kind,seed)
    cs,cr,ns=collect_counts(m)
    rows=[m.blocks[l].moe.gate.weight.detach().float().cpu().numpy() for l in range(N_LAYER)]
    rng=np.random.default_rng(seed)
    per={}
    for l in range(N_LAYER):
        g_s=top_experts(cs[l],GROUP); g_r=top_experts(cr[l],GROUP)
        g_x=sorted(rng.choice(N_EXP,size=GROUP,replace=False).tolist())
        c_s,c_r,c_x=coherence(rows[l],g_s),coherence(rows[l],g_r),coherence(rows[l],g_x)
        COH["STRUKTUR"].append((c_s,c_x)); COH["ZUFALLS-POS"].append((c_r,c_x))
        COH["ZUFALLS-EXP"].append((c_x,c_x))
        per.setdefault("s",[]).append(c_s/max(c_x,1e-12)); per.setdefault("r",[]).append(c_r/max(c_x,1e-12))
    BYOPT.setdefault(kind,[]).extend(per["s"])
    print("  %s/seed%d: Struktur-Positionen=%d | Kohaerenz-Verhaeltnisse je Layer  Struktur %s | Zufalls-Pos %s"
          %(kind,seed,ns,["%.2f"%v for v in per["s"]],["%.2f"%v for v in per["r"]]))
# ---------------- Auswertung ------------------------------------------------
def stat(name):
    pairs=COH[name]; nplus=sum(1 for a,b in pairs if a>b)
    ratio=float(np.median([a/max(b,1e-12) for a,b in pairs]))
    return nplus,len(pairs),sign_p(nplus,len(pairs)),ratio
print("\nERGEBNIS (je Lauf x Layer = %d Instanzen, Vergleich gegen ZUFALLS-EXP):"%len(COH["STRUKTUR"]))
res={}
for nm in ("STRUKTUR","ZUFALLS-POS"):
    k,n_,p,r=stat(nm); res[nm]=(p,r)
    print("  %-12s kohaerenter in %d/%d Instanzen | p=%.4f | medianes Verhaeltnis %.3f"%(nm,k,n_,p,r))
code=verdict_lab(res["STRUKTUR"][0],res["STRUKTUR"][1],res["ZUFALLS-POS"][0],res["ZUFALLS-POS"][1])
mu_r=float(np.median(BYOPT.get("muon",[np.nan]))); ad_r=float(np.median(BYOPT.get("adam",[np.nan])))
print("  Optimierer (Struktur-Verhaeltnis, median): Muon %.3f | Adam %.3f"%(mu_r,ad_r))
print("\nVERDIKT:",end=" ")
if code=="ARTEFAKT":
    print("SELEKTIONS-ARTEFAKT: auch ko-aktive Experten einer ZUFAELLIGEN Positions-")
    print("  menge sind kohaerenter als Zufalls-Experten (Verhaeltnis %.2f, p=%.4f)."%(res["ZUFALLS-POS"][1],res["ZUFALLS-POS"][0]))
    print("  Ko-Aktivierung erzeugt Router-Kohaerenz per Konstruktion - der Cell-27A-")
    print("  Befund (+7.8%) ist damit als Artefakt zu fuehren, nicht als Modul-Signatur.")
elif code=="ECHT":
    print("BEFUND HAELT: nur die STRUKTURIERT ko-aktiven Experten sind kohaerenter")
    print("  (%.2f, p=%.4f), die zufalls-positionsbasierten nicht (%.2f, p=%.3f)."
          %(res["STRUKTUR"][1],res["STRUKTUR"][0],res["ZUFALLS-POS"][1],res["ZUFALLS-POS"][0]))
    print("  Ko-Aktivierung allein genuegt nicht - Cell 27A misst echte Struktur.")
elif code=="INVERS":
    print("INVERS: die Zufalls-Positionen sind kohaerent, die strukturierten nicht -")
    print("  unerwartet; Zaehlung und Gruppenbildung pruefen, keine Aussage.")
else:
    print("KEIN-EFFEKT: in diesem Spielzeug-MoE entsteht ueberhaupt keine Gruppen-")
    print("  Kohaerenz (Struktur %.2f p=%.3f, Zufalls-Pos %.2f p=%.3f). Das Labor"
          %(res["STRUKTUR"][1],res["STRUKTUR"][0],res["ZUFALLS-POS"][1],res["ZUFALLS-POS"][0]))
    print("  repliziert die Bedingung nicht - Cell 27A bleibt unentschieden (Groessen-")
    print("  unterschied 32 vs 256 Experten, 4 vs 40 Layer, Spielzeugdaten).")
fig,ax=plt.subplots(figsize=(7,4))
for i,nm in enumerate(("STRUKTUR","ZUFALLS-POS","ZUFALLS-EXP")):
    vals=[a for a,_ in COH[nm]]
    ax.scatter(np.full(len(vals),i)+np.linspace(-.12,.12,len(vals)),vals,
               s=28,alpha=.8,color=["#DC2626","#2563EB","#9CA3AF"][i])
    ax.plot([i-.2,i+.2],[np.median(vals)]*2,color="k",lw=2)
ax.set_xticks(range(3)); ax.set_xticklabels(["STRUKTUR","ZUFALLS-POS","ZUFALLS-EXP"])
ax.set_ylabel("Router-Paar-Kohaerenz (median |cos|)")
ax.set_title("Ist Router-Kohaerenz ein Selektions-Artefakt? (%d Laeufe x %d Layer)"%(len(RUNS),N_LAYER))
plt.tight_layout(); plt.show()
LAB_RESULTS=dict(verdict=code,struct=res["STRUKTUR"],randpos=res["ZUFALLS-POS"],
                 opt=(mu_r,ad_r),cfg=dict(n_exp=N_EXP,top_k=TOP_K,group=GROUP,steps=STEPS))
